## Local Inference on GPU 
Model page: https://huggingface.co/NAMAA-Space/Qari-OCR-0.4.0-VL-4B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/NAMAA-Space/Qari-OCR-0.4.0-VL-4B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 52.5 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Loading model in 16-bit precision (this takes a couple of minutes)...")

# 1. Load the model in half-precision (float16) to fit inside Kaggle's GPU VRAM
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "MBZUAI/AIN", 
    torch_dtype=torch.float16, 
    device_map="auto"
)

# 2. Limit the maximum pixels to prevent high-resolution screenshots from crashing the memory
min_pixels = 256 * 28 * 28
max_pixels = 1024 * 28 * 28 

# 3. Load the processor with the pixel limits applied
processor = AutoProcessor.from_pretrained(
    "MBZUAI/AIN", 
    min_pixels=min_pixels, 
    max_pixels=max_pixels
)

print("Model and processor successfully loaded into memory!")

Loading model in 16-bit precision (this takes a couple of minutes)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Model and processor successfully loaded into memory!


In [4]:
import torch
from qwen_vl_utils import process_vision_info

image_path = "/kaggle/input/datasets/aminuxx/legalll/your_screenshot.png.png" 

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_path,
            },
            # Stricter prompt demanding the full text
            {"type": "text", "text": "استخرج النص الكامل من هذه الصورة حرفياً من البداية إلى النهاية. لا تختصر أي شيء."},
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Generate with higher limits
with torch.no_grad(): 
    generated_ids = model.generate(
        **inputs, 
        max_new_tokens=2048,           # Doubled the output length
        repetition_penalty=1.05        # Prevents it from getting "stuck"
    )

generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("\n--- EXTRACTED TEXT ---\n")
print(output_text[0])


--- EXTRACTED TEXT ---

الفصل الثالث عشر
تحصيل الضرائب
المادة الثامنة والستون: استقطاع الضريبة
أ. يجب على كل مقيم سواء كان مكلفاً أو غير مكلف بمقتضى هذا النظام، وعلى المنشأة الدائمة في
المملكة لغير مقيم، ممن يدفعون مبلغ ما لغير مقيم من مصدر في المملكة استقطاع ضريبة من
المبلغ المدفوع وفقاً للأسعار الآتية:
%5
1. إيجار
%15
2. أثواب أو ربع
%20
3. أتعاب إدارة
%5
4. دفعات مقابل تذاكر طيران أو شحن جوي أو بحري
%5
5. دفعات مقابل خدمات اتصالات هاتفية دولية
%15
6. أي دفعات أخرى تحددها اللائحة على آلاً يتجاوز سعر الضريبة
في حالة المبالغ المدفوعة من قبل شخص طبيعي تنطبق شروط الاستقطاع التي تقضي بها هذه المادة
على الدفعات الخاصة بالنشاط لهذا الشخص.
ب. يجب على الشخص الذي يستقطع الضريبة بمقتضى هذه المادة الالتزام بما يأتي:
1. التسجيل لدى الهيئة وتسديد المبلغ المستقطع للهيئة خلال العشرة أيام الأولى من الشهر
الذي يلي الشهر الذي تم الدفع فيه للمستفيد.
2. تزويد المستفيد بشهادة تبين المبلغ المدفوع له وقيمة الضريبة المستقطعة.
3. تزويد الهيئة في نهاية السنة الضريبية باسم وعنوان ورقم المستفيد (الرقم المميز) إ